# Notebook B — Evaluation Harness (Final)
**Person B (Aneesh) | CS 590NN — Reconsolidation-Inspired Targeted Unlearning**

This notebook fills in the `over_extinction` field left null by Person A's circuit pipeline,
computes all evaluation metrics, generates figures, and writes final output files.

**Key toggle:** Set `DRY_RUN = True` (default) to run on Mac CPU with synthetic OE values.
Set `DRY_RUN = False` on a GPU machine for real inference (~45 min on T4).

---
## Section 1 — Configuration

In [ ]:
# ─── DRY_RUN toggle ────────────────────────────────────────────────────────────
# True  → skip OurMethod live inference; synthetic OE (CPU, <60s)
# False → run full transformer_lens inference loop (~45 min on Colab T4)
#
# Note: ROME and MEMIT rows ALWAYS carry Person C's live-inference numbers
# regardless of DRY_RUN. Only OurMethod is affected by the toggle.
DRY_RUN = True

# ─── Model & device ────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-0.6B"
DEVICE   = "cpu"  # change to "cuda" when DRY_RUN=False

# ─── Paths ─────────────────────────────────────────────────────────────────────
import os
# Jupyter-safe: __file__ is undefined in notebook kernels; use CWD instead.
try:
    NB_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NB_DIR = os.getcwd()
ROOT = os.path.abspath(os.path.join(NB_DIR, ".."))
# Sanity-check: if CWD is already the repo root, NB_DIR/.. goes one level too high.
if not os.path.isdir(os.path.join(ROOT, "circuit_pipeline")):
    ROOT = NB_DIR
RESULTS_DIR   = os.path.join(ROOT, "circuit_pipeline", "results")
FIGURES_DIR   = os.path.join(ROOT, "metrics_harness", "figures")
BASELINES_DIR = os.path.join(ROOT, "baselines")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Person A inputs ─────────────────────────────────────────────────────────────
CIRCUIT_LOG   = os.path.join(RESULTS_DIR, "week2_circuit_log.json")
ABLATION_JSON = os.path.join(RESULTS_DIR, "week3_ablation.json")
HARNESS_JSON  = os.path.join(RESULTS_DIR, "week3_harness_output.json")

# ── Person C baselines (source of truth) ───────────────────────────────────────
ROME_JSON             = os.path.join(BASELINES_DIR, "rome_full_metrics.json")
MEMIT_JSON            = os.path.join(BASELINES_DIR, "memit_full_metrics.json")
MERGED_BASELINES_JSON = os.path.join(BASELINES_DIR, "week3_harness_output_with_baselines.json")

# ── Person B outputs ────────────────────────────────────────────────────────────
FILLED_JSON  = os.path.join(RESULTS_DIR, "week3_harness_output_filled.json")
RESULTS_CSV  = os.path.join(RESULTS_DIR, "results_final.csv")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary_B_final.json")
SCHEMA_JSON  = os.path.join(RESULTS_DIR, "results_schema.json")

SCHEMA_VERSION = "1.1"

print(f"DRY_RUN        = {DRY_RUN}")
print(f"MODEL_ID       = {MODEL_ID}")
print(f"ROOT           = {ROOT}")
print(f"RESULTS_DIR    = {RESULTS_DIR}")
print(f"BASELINES_DIR  = {BASELINES_DIR}")
assert os.path.isdir(RESULTS_DIR),  f"RESULTS_DIR not found: {RESULTS_DIR}"
assert os.path.isdir(BASELINES_DIR), f"BASELINES_DIR not found: {BASELINES_DIR}"
for p in (ROME_JSON, MEMIT_JSON, MERGED_BASELINES_JSON, CIRCUIT_LOG, ABLATION_JSON, HARNESS_JSON):
    assert os.path.isfile(p), f"missing input: {p}"
print(f"FIGURES_DIR    = {FIGURES_DIR}")
print(f"SCHEMA_VERSION = {SCHEMA_VERSION}")
if DRY_RUN:
    print("\n⚠️  DRY_RUN=True — OurMethod OE is STRUCTURAL approximation.")
    print("   ROME/MEMIT rows carry Person C's live-inference numbers.")
    print("   Set DRY_RUN=False on GPU for OurMethod live OE + neighborhood metrics.")


In [ ]:
import json, csv, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='colorblind')
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    warnings.warn("seaborn not found — using matplotlib defaults")

try:
    from scipy.stats import spearmanr
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    warnings.warn("scipy not found — Spearman rho will be computed manually")

print("Imports complete")

---
## Section 2 — Harness Functions (Canonical Definitions)

All 7 functions are defined here with the same signatures as Notebook_B1_Metrics_Harness_Week1.

In [ ]:
# Canonical metric functions live in metrics_harness/harness_functions.py so
# the pytest suite (metrics_harness/tests/) can import them without loading the
# notebook. Re-bind them at module scope here so the notebook cells below can
# call them directly.
import sys
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

from harness_functions import (  # noqa: F401  (re-exported for notebook cells)
    compute_edit_success,
    compute_paraphrase_success,
    compute_over_extinction_bleed,
    compute_over_extinction_damage,
    compute_neighborhood_preservation,
    compute_locality_drop,
    run_harness,
    baseline_container,
    compute_pareto_frontier,
)

print("All 7 harness functions + ingest helpers imported from harness_functions.py")


In [ ]:
# ─── Unit tests ────────────────────────────────────────────────────────────────
assert compute_edit_success("The answer is French.", "French") == 1.0
assert compute_edit_success("The answer is English.", "French") == 0.0

nbhd = [{"bleed_new": True}, {"bleed_new": False}, {"bleed_new": True}]
assert abs(compute_over_extinction_bleed(nbhd) - 2/3) < 1e-9
assert compute_over_extinction_bleed([]) == 0.0

pre  = [{"correct_before": True},  {"correct_before": True},  {"correct_before": False}]
post = [{"correct_after":  False}, {"correct_after":  True},  {"correct_after":  True}]
assert abs(compute_over_extinction_damage(pre, post) - 0.5) < 1e-9

para = ["She speaks French.", "Her language is German.", "French is her mother tongue."]
assert abs(compute_paraphrase_success(para, "French") - 2/3) < 1e-9

nbhd_post = [{"correct_after": True}, {"correct_after": False}, {"correct_after": True}]
assert abs(compute_neighborhood_preservation(nbhd_post) - 2/3) < 1e-9

# Floating-point safe comparison (0.65 - 0.60 ≠ 0.05 exactly in IEEE 754)
assert abs(compute_locality_drop(0.65, 0.60) - 0.05) < 1e-9
assert compute_locality_drop(None, 0.60) is None

row = run_harness(
    method="test", model_id="model", idx=0, n_steps=1,
    gen_after="French is the answer", target_new="French",
    paraphrase_gens=["Speaks French.", "Speaks German."],
    neighborhood_results_post=[{"bleed_new": True}, {"bleed_new": False}],
)
assert row["edit_success"] == 1.0
assert row["paraphrase_success"] == 0.5
assert abs(row["over_extinction"] - 0.5) < 1e-6

print("All unit tests passed ✓")


---
## Section 3 — Load Data

In [ ]:
# ── Person A outputs ───────────────────────────────────────────────────────────
with open(HARNESS_JSON) as f:
    harness_data = json.load(f)
harness_rows = harness_data['rows']  # 150 rows, over_extinction=null

with open(CIRCUIT_LOG) as f:
    circuit_log = json.load(f)       # 50 entries: mlp_layers, n_mlp

with open(ABLATION_JSON) as f:
    ablation_data = json.load(f)     # {"1": [...], "5": [...], "10": [...]}

print(f"Harness rows (null OE): {len(harness_rows)}")
print(f"Circuit log entries:    {len(circuit_log)}")
print(f"Ablation step counts:   {list(ablation_data.keys())}")

# Build Person A lookups
n_mlp_map = {entry['idx']: entry['n_mlp'] for entry in circuit_log}
fp_true_map = {}
for n_steps_str, entries in ablation_data.items():
    n_steps = int(n_steps_str)
    for entry in entries:
        fp_true_map[(entry['idx'], n_steps)] = entry['final_p_true']
print(f"n_mlp lookup entries:   {len(n_mlp_map)}")
print(f"final_p_true entries:   {len(fp_true_map)}")

# ── Person C baselines ─────────────────────────────────────────────────────────
# baseline_container() (imported above) handles the "samples" vs "rows"
# container-key divergence between rome_full_metrics.json and memit_full_metrics.json.

with open(ROME_JSON) as f:
    rome_data = json.load(f)
with open(MEMIT_JSON) as f:
    memit_data = json.load(f)

rome_samples  = baseline_container(rome_data)
memit_samples = baseline_container(memit_data)
baseline_samples = {"ROME": rome_samples, "MEMIT": memit_samples}

# Aggregate metrics per baseline. ROME's aggregate dict lacks
# neighborhood_preservation and locality_drop, so we compute them from per-sample rows.
def _agg(samples: list, agg_metrics: dict) -> dict:
    def _avg(field):
        vals = [s.get(field) for s in samples if s.get(field) is not None]
        return round(sum(vals) / len(vals), 6) if vals else None
    return {
        "edit_success_rate":         agg_metrics.get("edit_success_rate", _avg("edit_success")),
        "paraphrase_success":        agg_metrics.get("paraphrase_success", _avg("paraphrase_success")),
        "over_extinction":           agg_metrics.get("over_extinction", _avg("over_extinction")),
        "neighborhood_preservation": agg_metrics.get("neighborhood_preservation", _avg("neighborhood_preservation")),
        "locality_drop":             agg_metrics.get("locality_drop", _avg("locality_drop")),
        "n_samples": len(samples),
    }

rome_agg  = _agg(rome_samples,  rome_data.get("metrics", {}))
memit_agg = _agg(memit_samples, memit_data.get("metrics", {}))

print(f"\nROME  (live): edit={rome_agg['edit_success_rate']:.4f}  "
      f"OE={rome_agg['over_extinction']:.4f}  "
      f"paraphrase={rome_agg['paraphrase_success']:.4f}  "
      f"preservation={rome_agg['neighborhood_preservation']:.4f}  n={rome_agg['n_samples']}")
print(f"MEMIT (live): edit={memit_agg['edit_success_rate']:.4f}  "
      f"OE={memit_agg['over_extinction']:.4f}  "
      f"paraphrase={memit_agg['paraphrase_success']:.4f}  "
      f"preservation={memit_agg['neighborhood_preservation']:.4f}  n={memit_agg['n_samples']}")


In [ ]:
# Load CounterFact — from ROME JSON (HF fallback available)
counterfact_samples = None
if os.path.exists(ROME_JSON):
    with open(ROME_JSON) as f:
        rome_json_data = json.load(f)
    counterfact_samples = rome_json_data.get('samples', None) or rome_json_data.get('results', None)
    if counterfact_samples:
        print(f"Loaded {len(counterfact_samples)} CounterFact samples from ROME JSON")

if counterfact_samples is None:
    try:
        from datasets import load_dataset
        cf = load_dataset("NeelNanda/counterfact-tracing", split="train")
        counterfact_samples = list(cf)
        print(f"Loaded {len(counterfact_samples)} CounterFact samples from HuggingFace")
    except Exception as e:
        print(f"CounterFact HF load failed: {e}")
        counterfact_samples = []
        print("Using circuit log prompts as fallback.")

# Load TruthfulQA — graceful skip if unavailable
truthfulqa_samples = []
try:
    from datasets import load_dataset
    tqa = load_dataset("truthful_qa", "generation", split="validation")
    truthfulqa_samples = list(tqa)
    print(f"Loaded {len(truthfulqa_samples)} TruthfulQA samples")
except Exception as e:
    print(f"TruthfulQA load skipped: {e}")
    print("TruthfulQA hold-out will use synthetic approximation.")

---
## Section 4 — Edit + Evaluate Loop (live inference, skipped when DRY_RUN=True)

In [ ]:
if not DRY_RUN:
    # ── Live inference path ─────────────────────────────────────────────────────
    try:
        from transformer_lens import HookedTransformer
        import torch
    except ImportError:
        raise ImportError("transformer_lens required for DRY_RUN=False. "
                          "pip install transformer_lens")

    def load_model_hookedtransformer(model_id: str, device: str):
        """Load model via transformer_lens."""
        model = HookedTransformer.from_pretrained(
            model_id,
            center_writing_weights=False,
            center_unembed=False,
            fold_ln=False,
            device=device,
        )
        model.eval()
        return model

    def get_circuit_params(model, mlp_layers: list[int]):
        """Extract trainable circuit parameters: W_in/W_out for each MLP layer."""
        params = []
        for layer_idx in mlp_layers:
            params.append(model.blocks[layer_idx].mlp.W_in)
            params.append(model.blocks[layer_idx].mlp.W_out)
        return params

    def get_first_token_id(model, text: str) -> int:
        """
        Return the token ID of the first token of `text` (space-prepended for
        mid-sentence targets). Handles multi-token targets safely — unlike
        to_single_token(), this never raises even for targets like 'Christianity'.
        """
        tokens = model.to_tokens(" " + text.strip(), prepend_bos=False)[0]
        return int(tokens[0].item())

    def contrastive_loss(model, prompt: str, new_id: int, true_id: int):
        """
        Contrastive loss: -log P(target_new first token) + log P(target_true first token).
        Matches the loss used in Notebook2_Ablation.ipynb.
        """
        tokens = model.to_tokens(prompt, prepend_bos=True)
        logits = model(tokens)[0, -1, :]
        log_probs = torch.log_softmax(logits, dim=-1)
        return -log_probs[new_id] + log_probs[true_id]

    # NOTE: The proposal's 4-step protocol requires a KL-divergence penalty in
    # Step 4 ("Re-stabilization: Re-freeze weights + KL constraint"). Person A's
    # Week 3 sprint task adds the KL term. When Person A confirms kl_weight > 0,
    # add it here:  loss += kl_weight * kl_divergence_on_holdout(model, holdout_tokens)
    KL_WEIGHT = 0.0  # TODO: set to Person A's final kl_weight once confirmed

    def run_edit_with_neighborhood_eval(
        model,
        sample: dict,
        mlp_layers: list[int],
        n_steps: int,
        neighborhood_prompts: list[str],
        target_new: str,
        kl_weight: float = KL_WEIGHT,
    ) -> dict:
        """
        4-step reconsolidation protocol:
          1. Save weights to CPU (labile state)
          2. Pre-edit neighborhood pass (baseline)
          3. n_steps contrastive gradient descent on circuit params (corrective signal)
          4. Post-edit neighborhood pass + restore weights (re-stabilization)
        Returns: over_extinction, oe_damage, neighborhood_results_pre/post, kl_final
        """
        params = get_circuit_params(model, mlp_layers)
        orig_weights = [p.data.clone().cpu() for p in params]

        # Pre-edit neighborhood pass
        model.eval()
        nbhd_pre = []
        target_true = sample.get('target_true', '')
        with torch.no_grad():
            for prompt in neighborhood_prompts:
                tokens = model.to_tokens(prompt, prepend_bos=True)
                # Use do_sample=False for deterministic greedy decoding
                gen = model.generate(tokens, max_new_tokens=5, do_sample=False)[0]
                gen_str = model.to_string(gen[tokens.shape[-1]:])
                nbhd_pre.append({
                    'prompt': prompt,
                    'gen_before': gen_str,
                    'target_true': target_true,
                    'target_new': target_new,
                    'correct_before': target_true.lower() in gen_str.lower()
                })

        # Edit: gradient descent on circuit params (contrastive loss + optional KL)
        model.train()
        optimizer = torch.optim.Adam(params, lr=0.005)
        edit_prompt = sample['prompt']
        # Use first-token IDs to handle multi-token targets (e.g., "Christianity", "Birmingham")
        new_id  = get_first_token_id(model, target_new)
        true_id = get_first_token_id(model, target_true)
        final_loss = 0.0
        for _ in range(n_steps):
            optimizer.zero_grad()
            loss = contrastive_loss(model, edit_prompt, new_id, true_id)
            # KL penalty placeholder — add when Person A confirms kl_weight
            # if kl_weight > 0:
            #     loss += kl_weight * kl_divergence_on_holdout(model, holdout_tokens)
            loss.backward()
            optimizer.step()
            final_loss = float(loss.item())

        # Post-edit neighborhood pass
        model.eval()
        nbhd_post = []
        with torch.no_grad():
            for prompt in neighborhood_prompts:
                tokens = model.to_tokens(prompt, prepend_bos=True)
                gen = model.generate(tokens, max_new_tokens=5, do_sample=False)[0]
                gen_str = model.to_string(gen[tokens.shape[-1]:])
                nbhd_post.append({
                    'prompt': prompt,
                    'gen_after': gen_str,
                    'target_true': target_true,
                    'target_new': target_new,
                    'bleed_new': target_new.lower() in gen_str.lower(),
                    'correct_after': target_true.lower() in gen_str.lower()
                })

        # Restore weights (re-stabilization)
        for p, w in zip(params, orig_weights):
            p.data.copy_(w.to(p.device))

        oe_bleed = compute_over_extinction_bleed(nbhd_post)
        oe_dmg   = compute_over_extinction_damage(nbhd_pre, nbhd_post)
        nbhd_pres = compute_neighborhood_preservation(nbhd_post)
        return {
            'over_extinction': oe_bleed,
            'oe_damage': oe_dmg,
            'neighborhood_preservation': nbhd_pres,
            'neighborhood_results_pre': nbhd_pre,
            'neighborhood_results_post': nbhd_post,
            'kl_final': final_loss,
        }

    print("Live inference functions loaded.")
    if KL_WEIGHT == 0.0:
        print("⚠️  KL_WEIGHT=0: KL penalty not active. Update KL_WEIGHT once Person A confirms.")
else:
    print("DRY_RUN=True — Section 4 (live inference) skipped.")

---
## Section 5 — Compute / Approximate OE for All 150 Rows

> ⚠️ **DRY_RUN=True: Synthetic OE values — NOT paper-quality**
>
> Formula: `oe_approx = edit_success × (1 − final_p_true) × (n_mlp/28) × 0.12`
>
> This encodes the hypothesis that stronger edits and wider circuits produce more bleed,
> capped conservatively at 12%. These values are **structural approximations only**.
>
> Set `DRY_RUN = False` (Section 1) on a GPU to compute real OE from model inference.

In [ ]:
oe_results = {}  # (idx, n_steps) -> {over_extinction, oe_damage, ...}
AVG_N_MLP = 20.44  # fallback from Notebook1 summary

if DRY_RUN:
    # Synthetic approximation: no GPU needed
    for row in harness_rows:
        idx     = row['idx']
        n_steps = row['n_steps']
        es      = row['edit_success']
        fpt     = fp_true_map.get((idx, n_steps), 0.0)
        n_mlp   = n_mlp_map.get(idx, AVG_N_MLP)
        oe_approx = es * (1.0 - fpt) * (n_mlp / 28.0) * 0.12
        oe_results[(idx, n_steps)] = {
            'over_extinction': round(oe_approx, 6),
            'oe_damage': None,  # requires live inference
            'neighborhood_preservation': None,
            'oe_source': 'synthetic_approx',
        }
    print(f"Computed synthetic OE for {len(oe_results)} (idx, n_steps) pairs")

else:
    # Live inference — run Section 4 loop
    print("Running live inference... (this may take ~45 minutes on T4)")
    model = load_model_hookedtransformer(MODEL_ID, DEVICE)
    sample_lookup = {s.get('idx', i): s for i, s in enumerate(counterfact_samples[:50])}

    for row in harness_rows:
        idx     = row['idx']
        n_steps = row['n_steps']
        if (idx, n_steps) in oe_results:
            continue
        sample = sample_lookup.get(idx, {})
        mlp_layers = circuit_log[idx]['mlp_layers'] if idx < len(circuit_log) else []
        neighborhood_prompts = sample.get('neighborhood_prompts', [])[:10]
        if not neighborhood_prompts:
            neighborhood_prompts = [sample.get('prompt', '')] * 5  # fallback
        result = run_edit_with_neighborhood_eval(
            model, sample, mlp_layers, n_steps,
            neighborhood_prompts, sample.get('target_new', '')
        )
        result['oe_source'] = 'live_inference'
        oe_results[(idx, n_steps)] = result
        if idx % 10 == 0:
            print(f"  idx={idx}, n_steps={n_steps}: OE={result['over_extinction']:.4f}")

    print(f"Live inference complete: {len(oe_results)} results")

In [ ]:
# Update all 150 harness rows with computed OE
for row in harness_rows:
    key = (row['idx'], row['n_steps'])
    result = oe_results.get(key, {})
    row['over_extinction']         = result.get('over_extinction', 0.0)
    row['oe_damage']               = result.get('oe_damage', None)
    row['neighborhood_preservation'] = result.get('neighborhood_preservation', None)
    row['oe_source']               = result.get('oe_source', 'synthetic_approx')
    row['n_mlp']                   = n_mlp_map.get(row['idx'], AVG_N_MLP)
    if 'paraphrase_success' not in row:
        row['paraphrase_success'] = None
    if 'locality_drop' not in row:
        row['locality_drop'] = None

# Verify 0 null OE values
null_oe = sum(1 for r in harness_rows if r['over_extinction'] is None)
assert null_oe == 0, f"Still {null_oe} null OE values!"
print(f"✓ 0 null OE values in {len(harness_rows)} rows")

if DRY_RUN:
    print("\n⚠️  WARNING: OE values are SYNTHETIC — run with DRY_RUN=False for paper results")

---
## Section 6 — OE_damage (Collateral Damage Rate)

In [ ]:
if DRY_RUN:
    # OE_damage requires pre-edit baseline inference — not available in DRY_RUN mode
    print("DRY_RUN=True: oe_damage = None for all rows.")
    print("Explanation: OE_damage measures which neighbors the base model had correct")
    print("  before editing that are broken by the edit. Computing it requires")
    print("  running the unedited model on all neighborhood prompts, which needs GPU inference.")
    for row in harness_rows:
        row['oe_damage'] = None
else:
    # oe_damage already computed in Section 5 live loop
    n_with_damage = sum(1 for r in harness_rows if r.get('oe_damage') is not None)
    avg_damage = (sum(r['oe_damage'] for r in harness_rows if r.get('oe_damage') is not None)
                  / max(n_with_damage, 1))
    print(f"OE_damage computed for {n_with_damage} rows, avg = {avg_damage:.4f}")

print("Section 6 complete.")

---
## Section 7 — Merge All Results & Write Outputs

In [ ]:
import csv as csv_module
from collections import defaultdict

# Build ablation lookup for final_p_true + training_loss_final
ablation_lookup = {}
for n_steps_str, entries in ablation_data.items():
    for entry in entries:
        ablation_lookup[(entry['idx'], int(n_steps_str))] = entry

# Merge final_p_true into each OurMethod row
for row in harness_rows:
    abl = ablation_lookup.get((row['idx'], row['n_steps']), {})
    row['final_p_true'] = abl.get('final_p_true', None)

# ── Filled JSON (150-row OurMethod companion, drop-in for week3_harness_output.json) ──
filled_output = {
    "model": harness_data['model'],
    "schema_version": SCHEMA_VERSION,
    "oe_source_note": (
        "synthetic_approx — DRY_RUN=True. Set DRY_RUN=False for paper-quality results."
        if DRY_RUN else "live_inference"
    ),
    "rows": harness_rows
}
with open(FILLED_JSON, 'w') as f:
    json.dump(filled_output, f, indent=2)
print(f"Wrote {FILLED_JSON} ({len(harness_rows)} rows)")

# ── Unified 250-row CSV (150 OurMethod + 50 ROME + 50 MEMIT) ───────────────────
# Column order: stable across runs; training_loss_final replaces kl_final (v1.0).
# baseline_prob and final_p_true added in v1.1.
CSV_COLS = [
    'method', 'model', 'idx', 'n_steps',
    'edit_success', 'over_extinction', 'oe_damage',
    'neighborhood_preservation', 'paraphrase_success', 'locality_drop',
    'baseline_prob', 'final_p_true', 'training_loss_final',
    'n_mlp', 'oe_source',
]

def _none_as_blank(v):
    return '' if v is None else v

csv_rows = []

# OurMethod (150 rows). kl_final in harness_rows is contrastive loss —
# rename to training_loss_final at write time.
for row in harness_rows:
    csv_rows.append({
        'method':                     row['method'],
        'model':                      row['model'],
        'idx':                        row['idx'],
        'n_steps':                    row['n_steps'],
        'edit_success':               row['edit_success'],
        'over_extinction':            row['over_extinction'],
        'oe_damage':                  _none_as_blank(row.get('oe_damage')),
        'neighborhood_preservation':  _none_as_blank(row.get('neighborhood_preservation')),
        'paraphrase_success':         _none_as_blank(row.get('paraphrase_success')),
        'locality_drop':              _none_as_blank(row.get('locality_drop')),
        'baseline_prob':              _none_as_blank(row.get('baseline_prob')),
        'final_p_true':               _none_as_blank(row.get('final_p_true')),
        'training_loss_final':        _none_as_blank(row.get('kl_final')),
        'n_mlp':                      _none_as_blank(row.get('n_mlp')),
        'oe_source':                  row.get('oe_source', 'synthetic_approx'),
    })

# Baselines (50 ROME + 50 MEMIT) — per-sample, tagged oe_source=baseline_live
for method_name, samples in baseline_samples.items():
    for s in samples:
        csv_rows.append({
            'method':                     method_name,
            'model':                      s.get('model', MODEL_ID),
            'idx':                        s.get('idx', -1),
            'n_steps':                    s.get('n_steps', 1),
            'edit_success':               s.get('edit_success'),
            'over_extinction':            s.get('over_extinction'),
            'oe_damage':                  _none_as_blank(s.get('oe_damage')),
            'neighborhood_preservation':  _none_as_blank(s.get('neighborhood_preservation')),
            'paraphrase_success':         _none_as_blank(s.get('paraphrase_success')),
            'locality_drop':              _none_as_blank(s.get('locality_drop')),
            'baseline_prob':              _none_as_blank(s.get('baseline_prob')),
            'final_p_true':               '',  # not produced by baselines
            'training_loss_final':        '',  # N/A for one-shot baselines
            'n_mlp':                      '',  # baselines edit at a fixed layer, no circuit
            'oe_source':                  'baseline_live',
        })

with open(RESULTS_CSV, 'w', newline='') as f:
    writer = csv_module.DictWriter(f, fieldnames=CSV_COLS)
    writer.writeheader()
    writer.writerows(csv_rows)

expected = 150 + sum(len(v) for v in baseline_samples.values())
assert len(csv_rows) == expected, f"Expected {expected} rows, got {len(csv_rows)}"
print(f"Wrote {RESULTS_CSV} ({len(csv_rows)} rows: 150 OurMethod + {len(rome_samples)} ROME + {len(memit_samples)} MEMIT)")

# ── Comparison table ───────────────────────────────────────────────────────────
def avg(vals):
    vals = [v for v in vals if v is not None]
    return round(sum(vals)/len(vals), 4) if vals else None

method_groups = defaultdict(list)
for row in harness_rows:
    method_groups[row['method']].append(row)

print("\n═══ Comparison Table ══════════════════════════════════════════════════════════════════")
print(f"{'Method':<22} {'edit_success':>13} {'OE_bleed':>10} {'preservation':>14} {'paraphrase':>12}")
print("─" * 82)
print(f"{'ROME (live)':<22} {rome_agg['edit_success_rate']:>13.4f} "
      f"{rome_agg['over_extinction']:>10.4f} "
      f"{rome_agg['neighborhood_preservation']:>14.4f} "
      f"{rome_agg['paraphrase_success']:>12.4f}")
print(f"{'MEMIT (live)':<22} {memit_agg['edit_success_rate']:>13.4f} "
      f"{memit_agg['over_extinction']:>10.4f} "
      f"{memit_agg['neighborhood_preservation']:>14.4f} "
      f"{memit_agg['paraphrase_success']:>12.4f}")
for method in ['OurMethod_1step', 'OurMethod_5step', 'OurMethod_10step']:
    rows_m = method_groups[method]
    tag = " [synth]" if DRY_RUN else ""
    print(f"{method + tag:<22} {avg([r['edit_success'] for r in rows_m]):>13.4f} "
          f"{avg([r['over_extinction'] for r in rows_m]):>10.4f} "
          f"{'TBD':>14} "
          f"{'TBD':>12}")
print("═" * 82)


---
## Section 8 — Circuit Scope vs OE Correlation

In [ ]:
def spearman_rho_manual(x, y):
    """Compute Spearman rho without scipy (rank-based, no tie correction)."""
    n = len(x)
    def ranks(arr):
        sorted_idx = sorted(range(n), key=lambda i: arr[i])
        r = [0.0] * n
        for rank, i in enumerate(sorted_idx):
            r[i] = float(rank + 1)
        return r
    rx, ry = ranks(x), ranks(y)
    d_sq = sum((rx[i] - ry[i])**2 for i in range(n))
    return 1 - 6 * d_sq / (n * (n**2 - 1))

print("Circuit scope (n_mlp) vs OE_bleed — Spearman correlations:")
print("─" * 45)
spearman_results = {}
for steps in [1, 5, 10]:
    subset = [r for r in harness_rows if r['n_steps'] == steps]
    nm  = [r['n_mlp'] for r in subset]
    oe  = [r['over_extinction'] for r in subset]
    if HAS_SCIPY:
        rho, p = spearmanr(nm, oe)
        rho, p = round(float(rho), 4), round(float(p), 4)
        print(f"  {steps:2d}-step: rho={rho:+.3f}, p={p:.4f}")
    else:
        rho = round(spearman_rho_manual(nm, oe), 4)
        p = None
        print(f"  {steps:2d}-step: rho={rho:+.3f} (p-value requires scipy)")
    spearman_results[steps] = {'rho': rho, 'p': p}

print("─" * 45)
print("Note: higher n_mlp → larger circuit → more potential for OE bleed.")
if DRY_RUN:
    print()
    print("⚠️  IMPORTANT: With DRY_RUN=True, the synthetic OE formula encodes n_mlp")
    print("   directly (oe ∝ n_mlp), so rho is MECHANICALLY INFLATED and not an")
    print("   empirical finding. Treat these correlations as structural placeholders only.")
    print("   Real correlations will differ — run with DRY_RUN=False to measure them.")

---
## Section 9 — Figures

In [ ]:
# Method ordering + labels used across all figures
METHODS_ORDERED = ['ROME', 'MEMIT', 'OurMethod_1step', 'OurMethod_5step', 'OurMethod_10step']
METHOD_LABELS   = ['ROME',  'MEMIT', 'Ours-1',          'Ours-5',          'Ours-10']
COLORS          = ['#4878d0', '#956cb4', '#ee854a', '#6acc65', '#d65f5f']

BASELINE_AGGS = {"ROME": rome_agg, "MEMIT": memit_agg}

def get_avg(method, field):
    """
    Return the aggregate value for (method, field).
    Baselines are sourced from Person C's aggregates (rome_agg, memit_agg).
    OurMethod is sourced from harness_rows (synthetic if DRY_RUN=True).
    Returns None if the field is unavailable (e.g. OurMethod neighborhood_preservation
    in DRY_RUN=True).
    """
    if method in BASELINE_AGGS:
        agg = BASELINE_AGGS[method]
        return (
            agg.get('edit_success_rate') if field == 'edit_success' else agg.get(field)
        )
    # OurMethod aggregate from harness_rows
    rows_m = [r for r in harness_rows if r['method'] == method]
    vals = [r.get(field) for r in rows_m if r.get(field) is not None]
    return sum(vals) / len(vals) if vals else None

title_sfx = " [OurMethod OE synthetic]" if DRY_RUN else ""

print("Generating figures...")


In [ ]:
# ── Figure 1: OE_bleed per method + edit_success overlay (all 5 methods) ───────
oe_vals = [get_avg(m, 'over_extinction') for m in METHODS_ORDERED]
es_vals = [get_avg(m, 'edit_success')    for m in METHODS_ORDERED]

x = np.arange(len(METHODS_ORDERED))
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.bar(x, oe_vals, color=COLORS, alpha=0.85, width=0.55, label='OE_bleed')
ax2.plot(x, es_vals, 'k-o', linewidth=1.8, markersize=8, label='Edit success')

# Annotate OE bars with values
for xi, v in zip(x, oe_vals):
    if v is not None:
        ax1.text(xi, v + max(oe_vals) * 0.02, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=9)

ax1.set_xlabel('Method', fontsize=12)
ax1.set_ylabel('Over-Extinction Bleed (OE)', fontsize=12)
ax2.set_ylabel('Edit Success Rate', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(METHOD_LABELS, fontsize=11)
ax1.set_ylim(0, max(oe_vals) * 1.25 + 0.02)
ax2.set_ylim(0, 1.12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.title(f'Over-Extinction Bleed vs Method{title_sfx}', fontsize=13, fontweight='bold')
fig.text(0.5, -0.03,
         'ROME/MEMIT: Person C live inference. OurMethod: DRY_RUN={}.'.format(DRY_RUN),
         ha='center', fontsize=8, style='italic', color='gray')
plt.tight_layout()
fig1_path = os.path.join(FIGURES_DIR, 'fig1_oe_vs_steps.png')
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig1_path}")


In [ ]:
# ── Figure 2: Circuit scope (n_mlp) vs OE_bleed, color=n_steps (OurMethod only) ──
# Baselines have no circuit (n_mlp=N/A), so they are excluded from this figure.
cmap = plt.get_cmap('plasma')
step_colors = {1: cmap(0.2), 5: cmap(0.55), 10: cmap(0.85)}

fig, ax = plt.subplots(figsize=(8, 5.5))
for steps in [1, 5, 10]:
    subset = [r for r in harness_rows if r['n_steps'] == steps]
    nm = [r['n_mlp'] for r in subset]
    oe = [r['over_extinction'] for r in subset]
    rho = spearman_results[steps]['rho']
    ax.scatter(nm, oe, alpha=0.65, s=55, color=step_colors[steps],
               edgecolor='white', linewidth=0.5,
               label=f'{steps}-step (ρ={rho:+.3f})')

ax.set_xlabel('Circuit Scope (n_mlp)', fontsize=12)
ax.set_ylabel('Over-Extinction Bleed (OE)', fontsize=12)
ax.legend(fontsize=10, title='OurMethod')
ax.set_title(f'Circuit Scope vs OE (OurMethod only){title_sfx}', fontsize=13, fontweight='bold')

if DRY_RUN:
    caption = ("CAVEAT: With DRY_RUN=True the synthetic OE formula includes n_mlp,\n"
               "so ρ is a structural prediction, not an empirical finding.")
else:
    caption = "Empirical correlation between discovered-circuit size and OE."
fig.text(0.5, -0.04, caption, ha='center', fontsize=8.5,
         style='italic', color='darkred' if DRY_RUN else 'gray')

plt.tight_layout()
fig2_path = os.path.join(FIGURES_DIR, 'fig2_mlp_scope_vs_oe.png')
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig2_path}")


In [ ]:
# ── Figure 3: Pareto — OE_bleed (lower=better) vs edit_success (higher=better) ──
# compute_pareto_frontier() is imported from harness_functions.py above.

# Assemble (method, OE_bleed, edit_success) points across all 5 methods
pareto_points = []
for m, lbl in zip(METHODS_ORDERED, METHOD_LABELS):
    oe = get_avg(m, 'over_extinction')
    es = get_avg(m, 'edit_success')
    pareto_points.append((m, oe, es))

pareto_methods = compute_pareto_frontier(pareto_points)
print(f"Pareto-optimal methods (minimize OE_bleed, maximize edit_success): {pareto_methods}")

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
for (m, oe, es), lbl, c in zip(pareto_points, METHOD_LABELS, COLORS):
    if oe is None or es is None:
        continue
    is_pareto = m in pareto_methods
    ax.scatter(oe, es,
               s=180 if is_pareto else 110,
               color=c,
               edgecolor='black' if is_pareto else 'none',
               linewidth=2 if is_pareto else 0,
               zorder=6 if is_pareto else 5,
               label=f'{lbl}{" ★" if is_pareto else ""}')
    ax.annotate(lbl, (oe, es), xytext=(7, 7), textcoords='offset points',
                fontsize=10, fontweight='bold' if is_pareto else 'normal')

# Draw the Pareto frontier as a step line through non-dominated points
frontier_pts = sorted(
    [(m, oe, es) for (m, oe, es) in pareto_points if m in pareto_methods and oe is not None and es is not None],
    key=lambda t: t[1]
)
if len(frontier_pts) >= 2:
    xs = [p[1] for p in frontier_pts]
    ys = [p[2] for p in frontier_pts]
    ax.plot(xs, ys, 'k--', alpha=0.4, linewidth=1.2, label='Pareto frontier', zorder=4)

ax.annotate('← Ideal', xy=(0.0, 1.0), xytext=(0.03, 0.96),
            fontsize=10, style='italic', color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

max_oe = max([p[1] for p in pareto_points if p[1] is not None])
ax.set_xlabel('Over-Extinction Bleed (OE) — lower is better', fontsize=12)
ax.set_ylabel('Edit Success Rate — higher is better', fontsize=12)
ax.set_title(f'Specificity–Generalization Pareto{title_sfx}', fontsize=13, fontweight='bold')
ax.set_xlim(-0.02, max_oe * 1.2 + 0.02)
ax.set_ylim(0, 1.12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

fig.text(0.5, -0.03,
         '★ = Pareto-optimal. Frontier connects non-dominated methods.',
         ha='center', fontsize=8.5, style='italic', color='gray')

plt.tight_layout()
fig3_path = os.path.join(FIGURES_DIR, 'fig3_pareto.png')
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig3_path}")


In [ ]:
# ── Figure 4: Method-ranking table (rendered as an image for the paper) ────────
fig4_fields = [
    ('edit_success',              'Edit Success',     'high'),
    ('over_extinction',           'OE_bleed',         'low'),
    ('neighborhood_preservation', 'Preservation',     'high'),
    ('paraphrase_success',        'Paraphrase Succ.', 'context'),
]

def _fmt(v):
    if v is None:
        return '—'
    return f'{v:.3f}'

# Build table data: rows = methods, cols = metrics
table_data = []
for m, lbl in zip(METHODS_ORDERED, METHOD_LABELS):
    row = [lbl]
    for field, _, _ in fig4_fields:
        v = get_avg(m, field)
        row.append(_fmt(v))
    table_data.append(row)

header = ['Method'] + [h for _, h, _ in fig4_fields]

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axis('off')
tbl = ax.table(
    cellText=table_data,
    colLabels=header,
    cellLoc='center',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)

# Highlight the best cell per column (after the Method column)
for col_idx, (field, _, direction) in enumerate(fig4_fields, start=1):
    col_vals = []
    for row_idx, m in enumerate(METHODS_ORDERED):
        v = get_avg(m, field)
        col_vals.append((row_idx, v))
    numeric = [(i, v) for i, v in col_vals if v is not None]
    if not numeric:
        continue
    if direction == 'high':
        best_row = max(numeric, key=lambda t: t[1])[0]
    elif direction == 'low':
        best_row = min(numeric, key=lambda t: t[1])[0]
    else:
        best_row = None
    if best_row is not None:
        cell = tbl[(best_row + 1, col_idx)]  # +1 because header is row 0
        cell.set_facecolor('#c6f5c6')

# Header row: bold + colored
for c in range(len(header)):
    tbl[(0, c)].set_facecolor('#3b3b3b')
    tbl[(0, c)].set_text_props(weight='bold', color='white')

title_sfx_local = title_sfx
plt.title(f'Method Ranking Across Metrics{title_sfx_local}', fontsize=13,
          fontweight='bold', pad=14)
fig.text(0.5, 0.02,
         'Green = best per column. OurMethod preservation/paraphrase require DRY_RUN=False.',
         ha='center', fontsize=8.5, style='italic', color='gray')

fig4_path = os.path.join(FIGURES_DIR, 'fig4_method_ranking.png')
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig4_path}")


---
## Section 10 — TruthfulQA Hold-out

In [ ]:
import re

# ── TruthfulQA hold-out: projected OE + lexical exposure rate ──────────────────
# We report two complementary signals without requiring GPU inference:
#
# 1. tqa_oe_projected_upper_bound (synthetic): a conservative projection
#    that assumes TruthfulQA OE is ≤ 80% of CounterFact OE due to the
#    partial domain overlap. Placeholder until DRY_RUN=False re-run.
#
# 2. tqa_exposure_rate (structural, GPU-free): for each method's edit set,
#    the fraction of TruthfulQA items whose question/reference text
#    lexically overlaps with any edited `target_new`. This measures
#    "how many TQA items are AT RISK from the edit set" — an honest
#    upper bound on potential hold-out OE without live inference.

OVERLAP_KEYWORDS = [
    'language', 'nationality', 'country', 'born', 'located', 'citizen',
    'native', 'speak', 'mother tongue', 'official', 'capital'
]

def _tokenize(s: str) -> set:
    return set(re.findall(r'[a-z]+', (s or '').lower()))

# Pull the 50 edited target_new strings from CounterFact (shared across methods
# since every method edits the same 50 samples).
edit_targets = []
if rome_samples:
    for s in rome_samples[:50]:
        tn = s.get('target_new')
        if tn:
            edit_targets.append(tn)
target_token_set = set()
for t in edit_targets:
    target_token_set |= _tokenize(t)

# Filter TruthfulQA by keyword overlap (same filter used previously)
tqa_filtered = []
if truthfulqa_samples:
    for sample in truthfulqa_samples:
        q = sample.get('question', '').lower()
        if any(kw in q for kw in OVERLAP_KEYWORDS):
            tqa_filtered.append(sample)
    print(f"TruthfulQA filtered: {len(tqa_filtered)} / {len(truthfulqa_samples)} samples")
else:
    print("TruthfulQA not loaded — lexical exposure will be None.")

# (2) Lexical exposure: fraction of TQA items whose combined (question +
# best_answer + correct_answers) tokens intersect the target token set.
def _tqa_exposure(items: list[dict], target_tokens: set) -> float:
    if not items or not target_tokens:
        return None
    hits = 0
    for it in items:
        txt_tokens = _tokenize(it.get('question', ''))
        ba = it.get('best_answer') or ''
        txt_tokens |= _tokenize(ba)
        for a in it.get('correct_answers') or []:
            txt_tokens |= _tokenize(a)
        if txt_tokens & target_tokens:
            hits += 1
    return round(hits / len(items), 4)

exposure_all      = _tqa_exposure(truthfulqa_samples, target_token_set)
exposure_filtered = _tqa_exposure(tqa_filtered,       target_token_set)

# (1) Projected upper bound by method
oe_bleed_cf = {m: get_avg(m, 'over_extinction') for m in METHODS_ORDERED}
tqa_oe_projected = {
    m: (round(oe * 0.8, 6) if oe is not None else None)
    for m, oe in oe_bleed_cf.items()
}

# Display
print(f"\n{'Method':<20} {'CF OE':>10} {'TQA OE (projected)':>22}")
print("─" * 55)
for m, lbl in zip(METHODS_ORDERED, METHOD_LABELS):
    cf = oe_bleed_cf[m]
    pj = tqa_oe_projected[m]
    cf_str = '—' if cf is None else f'{cf:10.6f}'
    pj_str = '—' if pj is None else f'{pj:22.6f}'
    print(f"{lbl:<20} {cf_str:>10} {pj_str:>22}")

print(f"\nTQA lexical exposure (all items):        {exposure_all}")
print(f"TQA lexical exposure (keyword-filtered): {exposure_filtered}")
print("Note: exposure is an upper bound on at-risk items, not a direct OE measurement.")


---
## Section 11 — Write Summary JSON + Paper Conclusions

In [ ]:
from datetime import date

def _round(v, ndigits=4):
    return None if v is None else round(v, ndigits)

summary = {
    "project": "Reconsolidation-Inspired Targeted Unlearning in LLMs",
    "model": MODEL_ID,
    "schema_version": SCHEMA_VERSION,
    "date": date.today().isoformat(),
    "ourmethod_oe_source": "synthetic_approx" if DRY_RUN else "live_inference",
    "baselines_oe_source": "baseline_live",
    "oe_note": (
        "DRY_RUN=True: OurMethod OE values are structural approximations. "
        "ROME/MEMIT numbers are from Person C's live inference. "
        "Run with DRY_RUN=False for OurMethod live OE + neighborhood metrics."
        if DRY_RUN else "All methods on live inference."
    ),
    "metric_clarification": (
        "Two complementary over-extinction metrics: "
        "(1) over_extinction / OE_bleed: fraction of neighborhood prompts where the "
        "NEW target bleeds in. ROME=0.693, MEMIT=0.513 per Person C's live run. "
        "(2) collateral_damage_rate = 1 - neighborhood_preservation: fraction of "
        "related facts disrupted. ROME=0.993, MEMIT=0.933 per Person C's per-sample "
        "rows. The proposal's 'collateral refusal' claim refers to metric (2). "
        "OurMethod's collateral_damage_rate can only be compared with DRY_RUN=False."
    ),
    "methods": {
        "ROME": {
            "edit_success":              _round(rome_agg['edit_success_rate']),
            "over_extinction_bleed":     _round(rome_agg['over_extinction']),
            "neighborhood_preservation": _round(rome_agg['neighborhood_preservation']),
            "collateral_damage_rate":    (_round(1.0 - rome_agg['neighborhood_preservation'])
                                          if rome_agg['neighborhood_preservation'] is not None else None),
            "paraphrase_success":        _round(rome_agg['paraphrase_success']),
            "locality_drop":             _round(rome_agg['locality_drop']),
            "n_samples":                 rome_agg['n_samples'],
            "oe_source":                 "baseline_live",
        },
        "MEMIT": {
            "edit_success":              _round(memit_agg['edit_success_rate']),
            "over_extinction_bleed":     _round(memit_agg['over_extinction']),
            "neighborhood_preservation": _round(memit_agg['neighborhood_preservation']),
            "collateral_damage_rate":    (_round(1.0 - memit_agg['neighborhood_preservation'])
                                          if memit_agg['neighborhood_preservation'] is not None else None),
            "paraphrase_success":        _round(memit_agg['paraphrase_success']),
            "locality_drop":             _round(memit_agg['locality_drop']),
            "n_samples":                 memit_agg['n_samples'],
            "oe_source":                 "baseline_live",
        },
        "OurMethod_1step": {
            "avg_edit_success":             _round(get_avg('OurMethod_1step', 'edit_success')),
            "avg_over_extinction_bleed":    _round(get_avg('OurMethod_1step', 'over_extinction')),
            "avg_neighborhood_preservation": "TBD — requires DRY_RUN=False",
            "n_samples":                    50,
            "oe_source":                    "synthetic_approx" if DRY_RUN else "live_inference",
        },
        "OurMethod_5step": {
            "avg_edit_success":             _round(get_avg('OurMethod_5step', 'edit_success')),
            "avg_over_extinction_bleed":    _round(get_avg('OurMethod_5step', 'over_extinction')),
            "avg_neighborhood_preservation": "TBD — requires DRY_RUN=False",
            "n_samples":                    50,
            "oe_source":                    "synthetic_approx" if DRY_RUN else "live_inference",
        },
        "OurMethod_10step": {
            "avg_edit_success":             _round(get_avg('OurMethod_10step', 'edit_success')),
            "avg_over_extinction_bleed":    _round(get_avg('OurMethod_10step', 'over_extinction')),
            "avg_neighborhood_preservation": "TBD — requires DRY_RUN=False",
            "n_samples":                    50,
            "oe_source":                    "synthetic_approx" if DRY_RUN else "live_inference",
        },
    },
    "pareto_methods": pareto_methods,
    "spearman_mlp_scope_vs_oe": {
        "_WARNING": ("DRY_RUN=True: rho is mechanically inflated (n_mlp is in the "
                     "synthetic OE formula). Not an empirical result.") if DRY_RUN else "live_inference",
        "1step":  {"rho": spearman_results[1]['rho'],  "p": spearman_results[1]['p']},
        "5step":  {"rho": spearman_results[5]['rho'],  "p": spearman_results[5]['p']},
        "10step": {"rho": spearman_results[10]['rho'], "p": spearman_results[10]['p']},
    },
    "tqa_oe_projected_upper_bound": {
        "_note": "Synthetic projection: 0.8 × counterfact OE_bleed. Placeholder until DRY_RUN=False run.",
        **{m: tqa_oe_projected[m] for m in METHODS_ORDERED},
    },
    "tqa_exposure_rate": {
        "_note": ("Fraction of TruthfulQA items whose question/answer text lexically "
                  "overlaps with any edit target_new. GPU-free upper bound on at-risk "
                  "items; NOT a direct OE measurement."),
        "all_items":        exposure_all,
        "keyword_filtered": exposure_filtered,
        "edit_target_token_count": len(target_token_set),
    },
    "todos_before_paper": [
        "Run with DRY_RUN=False on GPU to get OurMethod live OE_bleed and "
        "neighborhood_preservation (critical: needed to validate the paper's "
        "central claim that our method reduces collateral damage vs ROME/MEMIT).",
        "20-step ablation (proposal specifies 1/5/10/20; Person A ran 1/5/10 only). "
        "Requires GPU from Person A.",
        "KL penalty in edit loop. Accepted as OUT OF SCOPE per sprint contingency "
        "(L4 GPU OOMs on KL). Document as limitation in paper.",
        "Optional: ITI / LoRA / AlphaEdit baselines from Person C (sprint "
        "contingency allows dropping to 4-method comparison).",
        "Optional: scale to LLaMA-3-8B (sprint contingency allows staying on Qwen3-0.6B).",
    ],
    "key_finding": (
        f"On Qwen3-0.6B / 50 CounterFact samples, ROME and MEMIT achieve very high "
        f"edit success ({rome_agg['edit_success_rate']:.3f} and {memit_agg['edit_success_rate']:.3f}) "
        f"but suffer substantial OE_bleed ({rome_agg['over_extinction']:.3f} and "
        f"{memit_agg['over_extinction']:.3f}) AND catastrophic collateral damage "
        f"(preservation {rome_agg['neighborhood_preservation']:.3f} / "
        f"{memit_agg['neighborhood_preservation']:.3f}). OurMethod 5/10-step reaches "
        "comparable edit success with much lower synthetic OE_bleed (0.04-0.09). "
        "The Pareto frontier covers multiple non-dominated methods, confirming the "
        "expected specificity-generalization tradeoff. Whether OurMethod also "
        "REDUCES collateral damage (the proposal's 'over-extinction') vs. one-shot "
        "baselines requires DRY_RUN=False on GPU."
    )
}

with open(SUMMARY_JSON, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Wrote {SUMMARY_JSON}")
print(json.dumps(summary, indent=2))


### Paper Conclusions

**Key results (Qwen3-0.6B, 50 CounterFact samples):**

| Method | Edit Success | OE_bleed | Preservation | Collateral Damage | Paraphrase | Source |
|--------|:-------------:|:--------:|:------------:|:-----------------:|:----------:|:------:|
| ROME | **0.987** | 0.693 | 0.007 | **0.993** ← severe | 0.860 | live |
| MEMIT | 0.977 | 0.513 | 0.067 | **0.933** ← severe | 0.850 | live |
| OurMethod 1-step | 0.462 | ~0.040* | TBD | TBD | TBD | synth |
| OurMethod 5-step | 0.979 | ~0.086* | TBD | TBD | TBD | synth |
| OurMethod 10-step | **0.998** | ~0.087* | TBD | TBD | TBD | synth |

_\* OurMethod OE values are synthetic structural approximations in DRY_RUN=True mode._

**Metric clarification (important for paper framing):**

The proposal defines over-extinction as *"collateral refusal for associated subjects"* (sprint plan: *"collateral refusal rate on hold-out related facts"*). This maps to **collateral damage = 1 − neighborhood_preservation**. The 50-sample live runs show that one-shot baselines disrupt 93%–99% of neighborhood facts — ROME's 0.993 collateral-damage rate is the central over-extinction signal. This dwarfs the complementary OE_bleed metric (fraction of neighbors where the new answer spreads; ROME=0.693, MEMIT=0.513).

The paper's central claim — that circuit-targeted gradient editing reduces this collateral damage — can only be validated with a DRY_RUN=False run on GPU, since `neighborhood_preservation` for OurMethod requires pre- and post-edit neighborhood inference.

**Pareto analysis:**

Pareto-optimal methods on (OE_bleed, edit_success) axes are listed in `summary_B_final.json["pareto_methods"]` and visualized in `fig3_pareto.png`. In DRY_RUN=True synthetic mode the OurMethod 10-step variant appears Pareto-optimal against the baselines because its synthetic OE is structurally capped at ~0.09. The live-inference rerun will redraw this frontier.

**Qwen3-0.6B caveat:** This is a small model (0.6B parameters, 28 layers). Person A's ACDC pass selected MLP-only circuits (0 attention heads) across all 50 samples — likely a model-size artifact. Results here are a proof-of-concept; scaling to LLaMA-3-8B (sprint contingency: optional) is the natural follow-up.

**Accepted limitations (per sprint-doc section 5 contingencies):**

1. **KL penalty disabled.** Proposal Step 4 requires a KL-divergence stabilizer; L4 GPU (22 GB) OOMs on that extra forward pass, so KL is skipped. The `training_loss_final` column (renamed from `kl_final` in schema v1.1) carries the contrastive loss only.
2. **1/5/10-step ablation.** Proposal specifies 1/5/10/20; Person A ran 1/5/10 under the same GPU contingency.
3. **Baselines.** ROME and MEMIT are live; MEND declared infeasible (no Qwen3 hypernetwork checkpoint). ITI/LoRA/AlphaEdit are optional per contingency and not yet delivered.

**TruthfulQA hold-out (GPU-free signals in `summary_B_final.json`):**

- `tqa_oe_projected_upper_bound`: per-method conservative projection (0.8 × CounterFact OE). Placeholder.
- `tqa_exposure_rate`: fraction of TruthfulQA items whose question/reference text lexically overlaps with any edited target_new. An honest upper bound on at-risk items, not a direct OE measurement. Useful as a "which held-out items *could* break" signal.


In [ ]:
# ── Final verification ──────────────────────────────────────────────────────────
import os

EXPECTED_CSV_ROWS = 150 + sum(len(v) for v in baseline_samples.values())

checks = [
    (FILLED_JSON, "week3_harness_output_filled.json"),
    (RESULTS_CSV, "results_final.csv"),
    (SUMMARY_JSON, "summary_B_final.json"),
    (SCHEMA_JSON, "results_schema.json"),
    (os.path.join(FIGURES_DIR, 'fig1_oe_vs_steps.png'),    "fig1_oe_vs_steps.png"),
    (os.path.join(FIGURES_DIR, 'fig2_mlp_scope_vs_oe.png'), "fig2_mlp_scope_vs_oe.png"),
    (os.path.join(FIGURES_DIR, 'fig3_pareto.png'),          "fig3_pareto.png"),
    (os.path.join(FIGURES_DIR, 'fig4_method_ranking.png'),  "fig4_method_ranking.png"),
]

print("═══ Output File Verification ═══════════════════════════════════════════")
all_ok = True
for path, name in checks:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    status = "✓" if exists and size > 0 else "✗"
    print(f"  {status} {name:<42} {size:>10} bytes")
    if not exists or size == 0:
        all_ok = False

# Row count check
with open(RESULTS_CSV) as f:
    n_rows = sum(1 for _ in f) - 1
rows_ok = n_rows == EXPECTED_CSV_ROWS
print(f"\n  results_final.csv rows: {n_rows} (expected {EXPECTED_CSV_ROWS})"
      f" {'✓' if rows_ok else '✗'}")

# Null OE check
with open(FILLED_JSON) as f:
    filled = json.load(f)
null_oe = sum(1 for r in filled['rows'] if r.get('over_extinction') is None)
null_ok = null_oe == 0
print(f"  OurMethod null OE values: {null_oe} (expected 0) {'✓' if null_ok else '✗'}")

# Summary sanity check: ROME should show real numbers (not the stub 0.0 bug)
with open(SUMMARY_JSON) as f:
    summary_read = json.load(f)
rome_oe = summary_read['methods']['ROME']['over_extinction_bleed']
rome_ok = rome_oe is not None and rome_oe > 0.5  # real live ROME ≈ 0.693
print(f"  Summary ROME OE_bleed: {rome_oe} (expected ~0.693) {'✓' if rome_ok else '✗'}")

schema_ver = summary_read.get('schema_version')
schema_ok = schema_ver == SCHEMA_VERSION
print(f"  Summary schema_version: {schema_ver!r} (expected {SCHEMA_VERSION!r}) "
      f"{'✓' if schema_ok else '✗'}")

pareto = summary_read.get('pareto_methods', [])
pareto_ok = isinstance(pareto, list) and len(pareto) >= 1
print(f"  pareto_methods: {pareto} (expected ≥1 method) {'✓' if pareto_ok else '✗'}")

print("═══════════════════════════════════════════════════════════════════════")
if all_ok and rows_ok and null_ok and rome_ok and schema_ok and pareto_ok:
    print("✓ All verification checks passed.")
else:
    print("✗ Some checks failed — review output above.")
